# Lecture 2 — Interactive Maps with Python

**folium · contextily · Anzony Quispe · 2026**

## What you'll learn

- How to build **folium web maps** (Leaflet.js) directly from Python
- How to add **layer control** so viewers can toggle layers on/off
- How to group many points into **marker clusters** for a cleaner UX
- How to add **static basemaps with contextily** to matplotlib figures

## 0 · Setup

Install dependencies if needed (uncomment the line below).

In [ ]:
# !pip install geopandas folium contextily numpy pandas matplotlib pyogrio

In [ ]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["USE_PYGEOS"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
import contextily as cx
from shapely.geometry import Point

print("geopandas :", gpd.__version__)
print("folium    :", folium.__version__)
print("contextily:", cx.__version__)

## 1 · Download data & build variables

The cells below download the **Natural Earth 110m countries** shapefile and
construct all the variables the lecture examples depend on:
`world`, `south_america`, `cities`, `cities_m`, and `buf_ll`.

In [ ]:
import urllib.request, zipfile, tempfile

URL = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
tmp = tempfile.mkdtemp()
zip_path = os.path.join(tmp, "ne.zip")
urllib.request.urlretrieve(URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(tmp)

world = gpd.read_file(os.path.join(tmp, "ne_110m_admin_0_countries.shp"))
world = world[["NAME", "CONTINENT", "POP_EST", "geometry"]]
print("world loaded:", len(world), "countries")

south_america = world[world.CONTINENT == "South America"].copy()
print("south_america:", len(south_america), "countries")

In [ ]:
# Four Peruvian cities (lon, lat) — EPSG:4326
cities = gpd.GeoDataFrame(
    {"name": ["Lima", "Cusco", "Arequipa", "Iquitos"]},
    geometry=[
        Point(-77.0428, -12.0464),
        Point(-71.9675, -13.5320),
        Point(-71.5375, -16.3989),
        Point(-73.2536,  -3.7437),
    ],
    crs="EPSG:4326",
)

# Metric version (UTM 18S) + 50 km buffer column
cities_m = cities.to_crs(cities.estimate_utm_crs())
cities_m["buf_50km"] = cities_m.buffer(50_000)

# Buffer reprojected back to lon/lat for folium
buf_ll = cities_m.set_geometry("buf_50km").to_crs(4326)

print("cities:  ", list(cities["name"]))
print("cities_m CRS:", cities_m.crs)
print("buf_ll   CRS:", buf_ll.crs)

## 20 · Interactive maps with **folium**

`matplotlib` is great for static plots. For a **web map with multiple layers and a layer switcher**, use folium (a Python wrapper around Leaflet.js).

In [ ]:
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="cartodbpositron")

# Layer 1: country polygon
folium.GeoJson(
    world[world.NAME == "Peru"].__geo_interface__,
    name="Peru border",
    style_function=lambda f: {"color": "#444", "weight": 2, "fillOpacity": 0.05},
).add_to(m)

# Layer 2: city points with popups
cluster = MarkerCluster(name="Cities").add_to(m)
for _, r in cities.iterrows():
    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        popup=r["name"],
        icon=folium.Icon(color="red", icon="info-sign"),
    ).add_to(cluster)

# Layer 3: 50 km buffer polygons (transformed back to lon/lat)
buf_ll = cities_m.set_geometry("buf_50km").to_crs(4326)
folium.GeoJson(
    buf_ll.__geo_interface__,
    name="50 km buffers",
    style_function=lambda f: {"color": "orange", "fillColor": "orange", "fillOpacity": 0.2},
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## 21 · Static basemaps with **contextily**

Add an OpenStreetMap / Carto / satellite background to a matplotlib plot in one line.
The trick: reproject your data to **Web Mercator (EPSG:3857)** first.

In [ ]:
import contextily as cx

ax = cities.to_crs(3857).plot(figsize=(8, 8), color="red", markersize=80)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
ax.set_axis_off(); ax.set_title("Peruvian cities on OSM/Carto basemap")
plt.show()

## Note on data sources

All geographic data used in this notebook comes from **Natural Earth** (https://www.naturalearthdata.com), a public-domain map dataset.

| Variable | Source | Description |
|---|---|---|
| `world` | Natural Earth 110m cultural | 177 country polygons, EPSG:4326 |
| `south_america` | Subset of `world` | Countries where `CONTINENT == "South America"` |
| `cities` | Hand-coded (lon/lat) | Lima, Cusco, Arequipa, Iquitos — EPSG:4326 |
| `cities_m` | Reprojection of `cities` | Same cities in UTM 18S (metres), with a `buf_50km` column |
| `buf_ll` | Reprojection of `cities_m` buffers | 50 km buffers reprojected back to EPSG:4326 for folium |

The Natural Earth shapefile is downloaded automatically in Section 1 of this notebook.
No manual download is required.